# 1. From CSV contracts to comparable groups

**Question:** does grouping comparable contracts help transfer historical pricing errors?
This is the completed CSV extension, not the older Parquet/first-day prototype.
Read these notebooks in order: **01 data/clustering → 02 measurement → 03 prediction → 04 evaluation**.
They reuse the research functions rather than reimplementing the algorithms.

Default cells inspect the completed real-data run. Optional rebuild cells are disabled until
`RECOMPUTE=True`; all writes stay under `extension/notebooks/results/`. Source notebooks are
saved without execution outputs; executed review copies can be generated with `verify.py`.
Start each notebook in a fresh project `.venv` kernel and run top to bottom.

## 1.1 Locate the repository and completed experiment

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "extension/clustering/supervised/pipeline.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the prediction-market-analysis repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "extension/notebooks/results/.matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / "extension/notebooks/results/.cache"))

In [ ]:
import argparse

import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits

from extension.notebooks.support import (
    begin_stage,
    finish_stage,
    output_dir,
    published_paths,
    read_completed,
)

published, DATA, ASSIGNMENTS, STATS, RESULTS = published_paths(ROOT)
print("Completed experiment:", RESULTS.relative_to(ROOT))
print("CSV split design:", published.get("split", "See methodology"))

## 1.2 Configuration and lineage
The final model manifest identifies the actual CSV inputs, frozen assignments and audited
sufficient-statistics cache. We read those paths instead of silently substituting the full
historical Parquet dataset or manufacturing data. The legacy `src/` cache is read-only.
Changing grouping requires fresh downstream extraction because cache labels/fingerprints
must match; do not join new clusters to an old-label cache.

In [ ]:
RUN_NAME = "completion_walkthrough"
RECOMPUTE = False  # Explicit opt-in: expensive steps write isolated notebook artifacts.
USE_REBUILT_UPSTREAM = False  # True only after completing the preceding rebuild notebooks.

cluster_output = output_dir(ROOT, RUN_NAME, "clustering")
cluster_manifest = read_completed(ASSIGNMENTS.parent)
display(
    pd.DataFrame(
        [
            {"role": "CSV extract", "path": str(DATA)},
            {"role": "frozen assignments", "path": str(ASSIGNMENTS)},
            {"role": "audited statistics", "path": str(STATS)},
            {"role": "optional rebuild", "path": str(cluster_output)},
        ]
    )
)
display(pd.Series(cluster_manifest["config"], name="recorded clustering configuration"))

## 1.3 Preserve the supplied splits
Read actual market CSV membership, not a new random split. Events and contracts must not cross
splits. These are event-hash splits: recurring series and underlying outcomes can still cross.
The original data-preparation commands are documented in [scripts/README.md](../../scripts/README.md).
Raw metadata counts are deliberately distinct from downstream resolved/traded sample counts.

In [ ]:
from extension.clustering.supervised.models import validate_splits

membership = pd.concat(
    [
        pd.read_csv(DATA / "split" / split / "markets.csv", usecols=["ticker", "event_ticker"], keep_default_na=False)
        .rename(columns={"event_ticker": "event_id"})
        .assign(split=split)
        for split in ["train", "validation", "test"]
    ],
    ignore_index=True,
)
validate_splits(membership)
assert not membership.ticker.duplicated().any()
display(membership.groupby("split").agg(contracts=("ticker", "nunique"), events=("event_id", "nunique")))

## 1.4 Structural rules first
The rules separate domain, underlying subject and proposition type. For example, a price-above
question is not treated as a price-interval question. Rules are conservative heuristics, with
series fallbacks and unknown fields; they are not an authoritative taxonomy or sentiment model.
No outcome, final quote or final volume enters grouping. Below are actual recorded assignments
and a bounded check that the rule extractor reproduces their family labels.

In [ ]:
from extension.clustering.features import extract_attributes

columns = ["ticker", "event_ticker", "title", "yes_sub_title", "split", "family", "cluster", "status", "similarity"]
assignments = pd.read_csv(ASSIGNMENTS, usecols=columns, keep_default_na=False)
checked = assignments[["ticker", "split"]].merge(
    membership[["ticker", "split"]], on="ticker", validate="one_to_one", suffixes=("", "_csv")
)
assert checked.split.eq(checked.split_csv).all()
examples = assignments[assignments.status.eq("assigned")].head(8).copy()
examples["recomputed_family"] = [extract_attributes(row)["family"] for row in examples.to_dict("records")]
assert examples.family.eq(examples.recomputed_family).all()
display(examples[["title", "yes_sub_title", "family", "cluster", "recomputed_family"]])

## 1.5 Text refinement inside structural families
The offline representation fits word/bigram TF-IDF and up to 64 LSA dimensions on a deterministic
training sample of at most 20,000 contracts. Average-linkage cosine hierarchies are fitted within
families (at most 1,000 training contracts per family), with distance cutoff 0.35.
Validation/test contracts use frozen family centroids: similarity at least 0.75, at least 20
training contracts and 10 training events per supported cluster. Unsupported/unseen/distant
matches remain rejected. This is lexical similarity, not verified logical equivalence or sentiment.

The optional cells below expose the same staged functions as the prior notebook interface;
review mode reads the frozen result and does not refit it on held-out text/outcomes.

In [ ]:
from extension.clustering.pipeline import embed, group_contracts, load_contracts, write_reviews

if RECOMPUTE:
    config = dict(cluster_manifest["config"])
    config["data"], config["output"] = DATA, cluster_output
    if config.get("model"):
        config["model"] = Path(config["model"])
    cluster_args = argparse.Namespace(**config)
    rebuild = begin_stage(
        ROOT,
        cluster_output,
        "clustering",
        [DATA / "split" / split / "markets.csv" for split in ["train", "validation", "test"]],
        {k: str(v) if isinstance(v, Path) else v for k, v in config.items()},
    )
    with threadpool_limits(limits=2):
        contracts, attrition = load_contracts(DATA)
        vectors = embed(contracts, cluster_args)
        rebuilt = group_contracts(contracts, vectors, cluster_args)
else:
    print("Review mode: frozen training-fitted assignments are used; no embedding/model refit.")

In [ ]:
if RECOMPUTE:
    rebuilt.to_csv(cluster_output / "assignments.csv", index=False)
    write_reviews(rebuilt, cluster_output)
    finish_stage(cluster_output, rebuild, attrition=attrition)
    print("Rebuilt assignments:", cluster_output / "assignments.csv")
    print("Notebook 02 must opt into USE_REBUILT_UPSTREAM=True to use these labels.")

## 1.6 Coverage, rejected groups and human review
Coherence is not predictive accuracy. Show rejection coverage and examples before examining
outcomes; rejected records must remain visible. Use the earlier title viewer for deeper review.
The final supervised cohort applies stricter training support and an extra combination screen
later; the clustering counts here therefore differ from final predictive counts.

In [ ]:
display(assignments.groupby(["split", "status"]).size().rename("contracts").reset_index())
profiles = pd.read_csv(ASSIGNMENTS.parent / "cluster_profiles.csv")
display(profiles.head(12))
review = pd.read_csv(ASSIGNMENTS.parent / "review_examples.csv", keep_default_na=False)
display(review.head(10))

## What this stage establishes
We have a training-fitted map from contract wording/structure to groups, with frozen hold-out
assignment and explicit rejections. We have not established mispricing, predictability or profits.
Next: use eventual outcomes and executed costs to measure historical calibration.

References: [clustering methodology](../clustering/METHODOLOGY.md),
[prior staged notebook](../../notebooks/extension_prototype/02_csv_clustering.ipynb).